# 02 反事实攻击 (Counterfactual Attack)

## 实验概述

**目的：** 通过扰动输入新闻，检测模型是否依赖记忆而非文本分析。如果模型忽略输入的变化仍给出相同预测，则说明模型在"回忆"而非"分析"。

**两种反事实变体（用于泄露检测）：**
- `reverse_outcome`：翻转结论方向。非泄露模型应翻转预测。PC 高 = 泄露（模型忽略了方向反转）。
- `alter_numbers`：大幅修改数值至可能改变结论。非泄露模型应改变判断。PC 高 = 泄露（模型忽略了数字变化）。

**指标定义：**
- **PC（预测一致性）** = 原始与反事实预测相同的比例。高 = 泄露（模型忽略输入变化）
- **CI（置信度不变性）** = 1 - 平均置信度差异。趋近 1 = 泄露（置信度未随输入变化而调整）
- **IDS（输入依赖分数）** = 原始与反事实输出分布的 KL 散度。高 = 好（模型敏感于输入变化）
- **L（综合泄露分数）** = 0.4·PC + 0.3·CI - 0.3·IDS。越低越好。

**注意：** swap_entity 不在此 notebook 中评估。它测量的是实体名依赖性而非反事实鲁棒性，归入 notebook 03 的消融体系。

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.masking import extract_json_robust
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, load_counterfactual_variants
from src.experiment import run_counterfactual_attack
from src.masking import apply_masking
from src.prompts import scoring_prompt
from src.metrics import prediction_consistency, confidence_invariance, input_dependency_score, composite_leakage_score
from src.display_utils import show_comparison, show_llm_response, show_counterfactual_result
import json
import numpy as np
import pandas as pd

set_seed()
(PROJECT_ROOT / "data" / "results").mkdir(parents=True, exist_ok=True)

## 1. 加载数据

In [ ]:
test_cases = load_test_cases()
variants = load_counterfactual_variants()
print(f"Test cases: {len(test_cases)}")
print(f"Counterfactual variants: {len(variants)}")

# Index variants by (case_id, variant_type)
variant_map = {}
for v in variants:
    variant_map.setdefault(v.original_case_id, {})[v.variant_type] = v

## 2. 辅助函数：解析 LLM 响应

In [ ]:
def parse_response(raw: str, output_format: str = "5-bin") -> dict:
    """Parse LLM response into structured data."""
    data = extract_json_robust(raw)
    if not data:
        return {"direction": "neutral", "confidence": 0.5, "distribution": [20]*5}

    if output_format == "5-bin":
        dist = [
            data.get("strong_bear", 20),
            data.get("weak_bear", 20),
            data.get("neutral", 20),
            data.get("weak_bull", 20),
            data.get("strong_bull", 20),
        ]
        total = sum(dist)
        if total > 0:
            dist = [d / total for d in dist]
        else:
            dist = [0.2] * 5
        bull = dist[3] + dist[4]
        bear = dist[0] + dist[1]
        direction = "up" if bull > bear + 0.1 else ("down" if bear > bull + 0.1 else "neutral")
        confidence = max(bull, bear, dist[2])
        return {"direction": direction, "confidence": confidence, "distribution": dist}

    elif output_format == "binary":
        direction = data.get("direction", "neutral")
        return {"direction": direction, "confidence": 1.0 if direction != "neutral" else 0.5, "distribution": [0.2]*5}

    elif output_format == "scalar":
        score = float(data.get("score", 0))
        direction = "up" if score > 0.1 else ("down" if score < -0.1 else "neutral")
        return {"direction": direction, "confidence": abs(score), "distribution": [0.2]*5}

    return {"direction": "neutral", "confidence": 0.5, "distribution": [0.2]*5}

## 3. 运行反事实攻击

对每条测试用例，用原文和反事实变体分别送入 DeepSeek，收集响应。

In [ ]:
client = LLMClient()
output_format = "5-bin"

strategies = {
    "baseline": MaskingConfig(),
    "year_only": MaskingConfig(mask_year=True),
    "thales_v1": MaskingConfig(mask_year=True, role_play=True, cot_forced=True),
    "llm_mask": MaskingConfig(mask_year=True, mask_entity=True, mask_mode="llm"),
}

all_results = []

for strategy_name, config in strategies.items():
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name} ({config.label})")
    print(f"{'='*60}")

    orig_responses, cf_responses, task_meta = run_counterfactual_attack(
        client, config, test_cases, variant_map, output_format
    )

    for (tc, vt_name), orig_resp, cf_resp in zip(task_meta, orig_responses, cf_responses):
        orig_parsed = parse_response(orig_resp.raw_response, output_format)
        cf_parsed = parse_response(cf_resp.raw_response, output_format)

        all_results.append({
            "strategy": strategy_name,
            "case_id": tc.id,
            "variant_type": vt_name,
            "orig_direction": orig_parsed["direction"],
            "cf_direction": cf_parsed["direction"],
            "orig_confidence": orig_parsed["confidence"],
            "cf_confidence": cf_parsed["confidence"],
            "orig_distribution": orig_parsed["distribution"],
            "cf_distribution": cf_parsed["distribution"],
            "orig_raw": orig_resp.raw_response,
            "cf_raw": cf_resp.raw_response,
        })

print(f"\nTotal probe results: {len(all_results)}")

## 4. 计算 PC / CI / IDS 指标

In [ ]:
df = pd.DataFrame(all_results)

metrics_rows = []
for (strategy, vtype), group in df.groupby(["strategy", "variant_type"]):
    orig_dirs = group["orig_direction"].tolist()
    cf_dirs = group["cf_direction"].tolist()
    orig_confs = group["orig_confidence"].tolist()
    cf_confs = group["cf_confidence"].tolist()
    orig_dists = group["orig_distribution"].tolist()
    cf_dists = group["cf_distribution"].tolist()
    
    pc = prediction_consistency(orig_dirs, cf_dirs)
    consistent = [o == c for o, c in zip(orig_dirs, cf_dirs)]
    ci = confidence_invariance(orig_confs, cf_confs, consistent)
    ids = input_dependency_score(orig_dists, cf_dists)
    L = composite_leakage_score(pc, ci, ids)
    
    metrics_rows.append({
        "strategy": strategy,
        "variant_type": vtype,
        "PC": pc,
        "CI": ci,
        "IDS": ids,
        "L": L,
        "n": len(group),
    })

metrics_df = pd.DataFrame(metrics_rows)
print("PC / CI / IDS Results:")
print(metrics_df.to_string(index=False, float_format="%.3f"))

### 指标解读

上表中每行对应一种（策略, 变体类型）组合的泄露指标：

- **reverse_outcome**：PC 越低越好。低 PC 说明模型在结论被翻转后也翻转了预测——这是正常的"分析"行为。高 PC 说明模型无视方向反转仍给出相同预测——这是"记忆"的信号。
- **alter_numbers**：PC 越低越好。低 PC 说明模型对数字变化敏感。高 PC 说明模型忽略了数字，可能在做模式匹配。
- **IDS** 越高越好：表示原始和反事实的输出概率分布差异大，模型确实在根据不同输入做出不同判断。
- **L** 越低越好：综合泄露分数，负值意味着 IDS 贡献超过了 PC+CI，模型输出变化大。

## 5. 可视化

In [ ]:
import random

# 随机抽取展示：用结构化卡片替代 raw JSON（重新运行刷新样本）
tc_map = {tc.id: tc for tc in test_cases}

for vt in VariantType:
    vt_results = [r for r in all_results if r["strategy"] == "baseline" and r["variant_type"] == vt.value]
    samples = random.sample(vt_results, min(2, len(vt_results)))
    for r in samples:
        tc = tc_map.get(r["case_id"])
        orig_text = tc.news.content if tc else "(未找到原文)"
        cf_variant = variant_map.get(r["case_id"], {}).get(vt.value)
        cf_text = cf_variant.modified_content if cf_variant else "(未找到反事实文本)"
        show_counterfactual_result(
            case_id=r["case_id"],
            original_text=orig_text,
            cf_text=cf_text,
            orig_raw=r["orig_raw"],
            cf_raw=r["cf_raw"],
            variant_type=vt.value,
        )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, metric in enumerate(["PC", "CI", "IDS"]):
    pivot = metrics_df.pivot(index="strategy", columns="variant_type", values=metric)
    pivot.plot(kind="bar", ax=axes[i], rot=45)
    axes[i].set_title(f"{metric} by Strategy x Variant Type")
    axes[i].set_ylabel(metric)
    if metric in ("PC", "CI"):
        axes[i].set_ylim(0, 1)
    axes[i].legend(title="Variant", fontsize=8)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'counterfactual_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: composite leakage score
pivot_L = metrics_df.pivot(index="strategy", columns="variant_type", values="L")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_L, annot=True, fmt=".3f", cmap="RdYlGn_r", center=0.3)
plt.title("Composite Leakage L = 0.4*PC + 0.3*CI - 0.3*IDS")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'counterfactual_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. 与 Profit Mirage 对标

论文中美股结果参考（Table 1）：所有模型 PC > 0.69，FinMem PC = 0.8213

In [ ]:
print("=" * 60)
print("DeepSeek A-Share vs Profit Mirage US-Stock Benchmarks")
print("=" * 60)

# 只用 reverse_outcome 和 alter_numbers 计算（swap_entity 已移至 notebook 03）
COUNTERFACTUAL_TYPES = {vt.value for vt in VariantType}
baseline_metrics = metrics_df[
    (metrics_df["strategy"] == "baseline") &
    (metrics_df["variant_type"].isin(COUNTERFACTUAL_TYPES))
]
avg_pc = baseline_metrics["PC"].mean()
avg_ci = baseline_metrics["CI"].mean()
avg_ids = baseline_metrics["IDS"].mean()
avg_L = composite_leakage_score(avg_pc, avg_ci, avg_ids)

print(f"\nDeepSeek (A-Share, baseline, 反事实变体):")
print(f"  Avg PC  = {avg_pc:.3f}  (Profit Mirage: all models > 0.69, FinMem = 0.82)")
print(f"  Avg CI  = {avg_ci:.3f}")
print(f"  Avg IDS = {avg_ids:.3f}")
print(f"  Avg L   = {avg_L:.3f}")

# 分 variant_type 解读
for _, row in baseline_metrics.iterrows():
    vt = row["variant_type"]
    pc = row["PC"]
    if vt == "reverse_outcome":
        print(f"\n  reverse_outcome PC = {pc:.3f}：模型在 {pc:.0%} 的情况下忽略了方向反转，")
        if pc > 0.5:
            print(f"    说明约 {pc:.0%} 的预测可能基于记忆而非文本分析。")
        else:
            print(f"    说明模型多数情况下能正确响应方向变化，泄露程度较低。")
    elif vt == "alter_numbers":
        print(f"\n  alter_numbers PC = {pc:.3f}：模型在 {pc:.0%} 的情况下忽略了数字变化，")
        if pc > 0.7:
            print(f"    说明模型对数量信息不敏感，可能依赖模式匹配而非数字分析。")
        else:
            print(f"    说明模型对数字变化有一定敏感度。")

if avg_pc > 0.69:
    print(f"\n  >> 平均 PC 超过 Profit Mirage 基线 - A 股泄露程度与美股模型相当")
elif avg_pc > 0.5:
    print(f"\n  >> 中等 PC - 存在泄露但低于美股模型水平")
else:
    print(f"\n  >> 低 PC - 泄露程度低于美股模型水平")

# Save results
output = {"metrics": metrics_rows, "raw_results": all_results}
output_path = PROJECT_ROOT / 'data' / 'results' / 'counterfactual_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"\nResults saved to {output_path}")